# Day 02 · Explicit schemas and file ingestion

Day 01 read every column as text. Today we tell Spark the **exact type** of each column, and see what happens when a file contains bad data.

| Task | Concept |
|---|---|
| 1 | Explicit schemas: `StructType`, `StructField` and DDL strings |
| 2 | Why not let Spark guess (`inferSchema`) |
| 3 | Tricky CSV values: commas in quotes, blank vs `""`, empty files |
| 4 | Bad rows: PERMISSIVE, DROPMALFORMED, FAILFAST |
| 5 | `NOT NULL` in a schema is not a data check |
| 6 | CSV columns are matched by position, not by name |
| 7 | Audit columns: source file and load time |
| 8 | Reading nested JSON |

Run on the **pyspark-course** cluster, not Serverless. Cells marked ⚠️ are **meant** to fail: read the error, then continue.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json

RAW = "/Volumes/dbw_pyspark_course/retail/raw"

# A separate volume for the small test files we create today
spark.sql("CREATE VOLUME IF NOT EXISTS dbw_pyspark_course.retail.scratch")
FIX = "/Volumes/dbw_pyspark_course/retail/scratch/day02"

# Expected row count of each table
manifest = json.load(open(f"{RAW}/manifest.json"))
print(manifest["raw_counts"])

### Task 1 · Explicit schemas

**Problem:** Define the schema of all six tables using the types from `DATA_DICTIONARY.md`:

| Kind of column | Type |
|---|---|
| IDs and names | `STRING` |
| Money (`unit_price`, `list_price`, `cost_price`) | `DECIMAL(12,2)` |
| `discount_pct` | `DECIMAL(5,2)` |
| Quantities | `INT` |
| `signup_date` | `DATE` |
| `order_ts`, `return_ts`, `updated_at` | `TIMESTAMP` |

Then read each table with its schema and compare the row counts with the manifest.

**Expected:**
- Counts: customers 80 · stores 5 · products 20 · orders 300 · order_items 785 · returns 30.
- In `order_items`: `quantity: integer`, `unit_price: decimal(12,2)`, `discount_pct: decimal(5,2)`.
- `order_ts` is a real timestamp: `2026-07-01 09:00:00`.

In [0]:
# StructType = the shape of a whole row
# StructField = one column: (name, type, can it be null?)
customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(),    True),
    T.StructField("name",        T.StringType(),    True),
    T.StructField("email",       T.StringType(),    True),
    T.StructField("city",        T.StringType(),    True),
    T.StructField("signup_date", T.DateType(),      True),
    T.StructField("updated_at",  T.TimestampType(), True),
])

In [0]:
# The same idea written as a DDL string: shorter and easier to read
stores_schema      = "store_id STRING, store_name STRING, city STRING"
products_schema    = "product_id STRING, product_name STRING, category STRING, list_price DECIMAL(12,2), cost_price DECIMAL(12,2)"
orders_schema      = "order_id STRING, customer_id STRING, store_id STRING, order_ts TIMESTAMP, status STRING, channel STRING"
order_items_schema = "line_id STRING, order_id STRING, product_id STRING, quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)"
returns_schema     = "return_id STRING, line_id STRING, return_quantity INT, return_ts TIMESTAMP, reason STRING"

In [0]:
# .schema(...) tells Spark the types, so it doesn't have to guess
customers   = spark.read.option("header", True).schema(customers_schema).csv(f"{RAW}/customers.csv")
stores      = spark.read.option("header", True).schema(stores_schema).csv(f"{RAW}/stores.csv")
products    = spark.read.option("header", True).schema(products_schema).csv(f"{RAW}/products.csv")
orders      = spark.read.option("header", True).schema(orders_schema).csv(f"{RAW}/orders.csv")
order_items = spark.read.option("header", True).schema(order_items_schema).csv(f"{RAW}/order_items.csv")
returns     = spark.read.option("header", True).schema(returns_schema).csv(f"{RAW}/returns.csv")

order_items.printSchema()
orders.show(3)

In [0]:
# Compare each table's row count with the manifest
print("customers  :", customers.count(),   "| expected", manifest["raw_counts"]["customers"])
print("stores     :", stores.count(),      "| expected", manifest["raw_counts"]["stores"])
print("products   :", products.count(),    "| expected", manifest["raw_counts"]["products"])
print("orders     :", orders.count(),      "| expected", manifest["raw_counts"]["orders"])
print("order_items:", order_items.count(), "| expected", manifest["raw_counts"]["order_items"])
print("returns    :", returns.count(),     "| expected", manifest["raw_counts"]["returns"])

### Task 2 · Why not let Spark guess the types?

**Problem:**
1. Read `order_items.csv` with `inferSchema=True`, so Spark guesses the types.
2. Read a file of store codes such as `007`, first by guessing and then with an explicit `STRING`.
3. Compare `0.1 + 0.2` as `DOUBLE` and as `DECIMAL`.

**Expected:**
1. The guesses: `unit_price: integer` (the file has `700`, not `700.00`) and `discount_pct: double`. Spark also has to **read the whole file once just to guess**, which is an extra job.
2. Guessing turns `007` into **7**, and the zeros are lost. With `STRING`, `007` stays `007`.
3. As a double: `0.30000000000000004`. As a decimal: exactly `0.3`. That's why money uses `DECIMAL`.

In [0]:
# Let Spark guess the types
guessed = spark.read.option("header", True).option("inferSchema", True).csv(f"{RAW}/order_items.csv")
guessed.printSchema()

In [0]:
# A small test file with codes that start with zeros
dbutils.fs.put(f"{FIX}/store_codes.csv", "store_code,pincode\n007,560001\n042,110001\n", overwrite=True)

# Guessed types: 007 becomes the number 7
spark.read.option("header", True).option("inferSchema", True).csv(f"{FIX}/store_codes.csv").show()

# Explicit STRING: 007 stays 007
spark.read.option("header", True).schema("store_code STRING, pincode STRING").csv(f"{FIX}/store_codes.csv").show()

In [0]:
# 0.1D is a DOUBLE (floating point); a plain 0.1 is a DECIMAL in Spark SQL
spark.sql("SELECT 0.1D + 0.2D AS as_double, 0.1 + 0.2 AS as_decimal").show()

### Task 3 · Tricky CSV values

**Problem:** Read a small file where:
- `C1`'s name has a comma inside quotes: `"Sharma, Asha"`
- `C2`'s email is blank
- `C3`'s email is `""` (quoted empty text)

Then read a file that has **only a header line**.

**Expected:**
- `C1`'s name is `Sharma, Asha`. The quotes protect the comma, so it isn't treated as a column separator.
- `C2`'s email is **null**, meaning "no value". `C3`'s email is **empty text**, meaning "a value that happens to be empty". They're different.
- The header-only file gives **0 rows**, but still the 3 columns from the schema.

In [0]:
# Three customers: a comma inside quotes, a blank email, and an email of "" (empty text)
dbutils.fs.put(f"{FIX}/tricky.csv",
    'customer_id,name,email\n'
    'C1,"Sharma, Asha",asha@example.test\n'
    'C2,Ravi,\n'
    'C3,Meena,""\n',
    overwrite=True)

tricky = spark.read.option("header", True).schema("customer_id STRING, name STRING, email STRING").csv(f"{FIX}/tricky.csv")
tricky.show()

In [0]:
# Is the email missing (null) or empty text? Easy rule: withColumn("column_name", expression) means create or replace that column using the given expression
tricky.withColumn("email_is_null", F.col("email").isNull()).show()

In [0]:
# A file with only the header line and no data
dbutils.fs.put(f"{FIX}/header_only.csv", "customer_id,name,email\n", overwrite=True)

empty = spark.read.option("header", True).schema("customer_id STRING, name STRING, email STRING").csv(f"{FIX}/header_only.csv")

print("rows   :", empty.count())
print("columns:", empty.columns)

### Task 4 · Bad rows: the three parse modes

**Problem:** A test file has three lines:
- `L1` is fine.
- `L2` has quantity `two`, which isn't a number.
- `L3` has a price, `12345678901234.56`, that is too big for `DECIMAL(12,2)` (at most 10 digits before the point).

Read it in each mode and compare.

**Expected:**
| Mode | What happens |
|---|---|
| **PERMISSIVE** (default) | All 3 rows are kept, and the bad values become null. The `_corrupt_record` column holds the original text of `L2` and `L3`. |
| **DROPMALFORMED** | Only `L1` is shown. **Surprise:** `count()` may still say 3, because Spark only checks the columns a query needs, and `count()` needs none. |
| **FAILFAST** | An error at the first bad row. It appears at `show()`, not when building the DataFrame (lazy evaluation). |

In [0]:
# Create the test file
dbutils.fs.put(f"{FIX}/bad_lines.csv",
    "line_id,quantity,unit_price\n"
    "L1,2,10.50\n"
    "L2,two,10.50\n"
    "L3,1,12345678901234.56\n",
    overwrite=True)

In [0]:
# PERMISSIVE: keep every row; bad values become null.
# The extra _corrupt_record column keeps the original text of each bad line.
# Because Spark has a default name for the corrupt-record column: _corrupt_record.When Spark reads a CSV file, it uses this name to identify where malformed records should be stored. If you rename the schema field to corrupted, Spark won't necessarily recognize it as the destination for those records unless you configure the new name.
permissive = (spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema("line_id STRING, quantity INT, unit_price DECIMAL(12,2), _corrupt_record STRING")
    .csv(f"{FIX}/bad_lines.csv"))

permissive.show(truncate=False)

In [0]:
# DROPMALFORMED: throw the bad rows away
dropped = (spark.read
    .option("header", True)
    .option("mode", "DROPMALFORMED")
    .schema("line_id STRING, quantity INT, unit_price DECIMAL(12,2)")
    .csv(f"{FIX}/bad_lines.csv"))

dropped.show()

In [0]:
# Surprise: count() may still say 3.
# Spark only parses the columns a query needs, and count() needs none, so no row is checked.
print("count():", dropped.count())

In [0]:
# FAILFAST: stop with an error at the first bad row.
# ⚠️ This cell is SUPPOSED to fail. Read the error, then continue with the next cell.
failfast = (spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema("line_id STRING, quantity INT, unit_price DECIMAL(12,2)")
    .csv(f"{FIX}/bad_lines.csv"))

failfast.show()   # the error appears here, at the action

### Task 5 · `NOT NULL` in a schema is not a data check

**Problem:** Declare `order_id` as `NOT NULL` and read `order_items.csv`, which has two lines with a blank `order_id`. Does Spark reject them?

**Expected:** **No.**
- `printSchema()` shows `order_id ... (nullable = true)`. When reading files, Spark switches every column back to nullable.
- Rows **`BAD08`** and **`BAD09`** still appear, with a null `order_id`.

Real data checks need explicit filters, which come on Day 04.

In [0]:
# Declare order_id as NOT NULL
strict = (spark.read
    .option("header", True)
    .schema("line_id STRING, order_id STRING NOT NULL, product_id STRING, "
            "quantity INT, unit_price DECIMAL(12,2), discount_pct DECIMAL(5,2)")
    .csv(f"{RAW}/order_items.csv"))

# Look: Spark changed order_id back to nullable = true
strict.printSchema()

# And the rows with a missing order_id are still there
strict.filter(F.col("order_id").isNull()).show()

### Task 6 · CSV columns are matched by position, not by name

**Problem:** A file arrives with its columns in a different order (`name,customer_id`) from your schema (`customer_id, name`). What does Spark do?

**Expected:**
- **The values get swapped, with no error.** `customer_id` shows `Asha` and `Ravi`. With an explicit schema, Spark fills columns by **position** and ignores the header names.
- `enforceSchema=False` makes Spark compare the header with the schema. It fails with `CSV header does not conform to the schema`, which is better than wrong data.

In [0]:
# The file has the columns in a different order: name first, then customer_id
dbutils.fs.put(f"{FIX}/reordered.csv", "name,customer_id\nAsha,C001\nRavi,C002\n", overwrite=True)

# Our schema says customer_id first. Spark matches by POSITION.
swapped = spark.read.option("header", True).schema("customer_id STRING, name STRING").csv(f"{FIX}/reordered.csv")
swapped.show()   # customer_id now contains names!

In [0]:
# enforceSchema=False: Spark checks that the header names match the schema.
# ⚠️ This cell is SUPPOSED to fail with "CSV header does not conform to the schema".
checked = (spark.read
    .option("header", True)
    .option("enforceSchema", False)
    .schema("customer_id STRING, name STRING")
    .csv(f"{FIX}/reordered.csv"))

checked.show()

### Task 7 · Audit columns

**Problem:** Add two columns to `orders`: the file each row came from, and the time it was loaded.

**Expected:**
- `source_file` is the full path ending in `.../raw/orders.csv`.
- `ingested_at` is the same timestamp on every row of this run.
- `_metadata` is a hidden column that every DataFrame read from files has. On Unity Catalog, use it rather than the older `input_file_name()`.

In [0]:
orders_audit = (orders
    .withColumn("source_file", F.col("_metadata.file_path"))   # which file the row came from
    .withColumn("ingested_at", F.current_timestamp()))         # when we loaded it

orders_audit.select("order_id", "source_file", "ingested_at").show(3, truncate=False)

### Task 8 · Reading nested JSON

**Before running:** go to **Catalog → dbw_pyspark_course → retail → raw**, click **Create directory**, name it `events_archive`, and upload `data/events_archive/batch_001.json` into it. You can upload all 6 batch files now, because the streaming days use them.

**Problem:** Each event has a **list** of items, a nested **device** record and a **map** of attributes. Read one file with an explicit nested schema and pull out one field from each.

**Expected:**
- **8 events:** 3 `view` events with 0 items and 5 `purchase` events with 2 items each.
- `os` is `android` and `campaign` is `autumn`.
- `event_time` is a real timestamp.

In [0]:
# One item inside an event
item_schema = T.StructType([
    T.StructField("product_id", T.StringType()),
    T.StructField("quantity",   T.IntegerType()),
    T.StructField("unit_price", T.DecimalType(12, 2)),
])

# The device record inside an event
device_schema = T.StructType([
    T.StructField("os",          T.StringType()),
    T.StructField("app_version", T.StringType()),
])

# The whole event (same as examples/schemas.py)
event_schema = T.StructType([
    T.StructField("event_id",    T.StringType()),
    T.StructField("event_time",  T.TimestampType()),
    T.StructField("customer_id", T.StringType()),
    T.StructField("order_id",    T.StringType()),
    T.StructField("event_type",  T.StringType()),
    T.StructField("items",       T.ArrayType(item_schema)),                    # a list of items
    T.StructField("device",      device_schema),                               # a record inside the record
    T.StructField("attributes",  T.MapType(T.StringType(), T.StringType())),   # key -> value pairs
])

In [0]:
events = spark.read.schema(event_schema).json(f"{RAW}/events_archive/batch_001.json")

events.printSchema()
events.show(truncate=False)

In [0]:
events.select(
    "event_id",
    "event_type",
    F.size("items").alias("number_of_items"),           # length of a list
    F.col("device.os").alias("os"),                     # field inside a record: use a dot
    F.col("attributes")["campaign"].alias("campaign"),  # value from a map: use ["key"]
).show()

### Key takeaways
- **Always give Spark a schema** for real pipelines. Guessing costs an extra read, loses leading zeros and can pick the wrong type (e.g. `double` for money).
- **StructType / StructField** and **DDL strings** describe the same thing. Use whichever is easier to read.
- **Blank vs `""`:** a blank field is null ("no value"), and a quoted `""` is empty text.
- **Parse modes:**
  - PERMISSIVE keeps rows and nulls the bad values; add `_corrupt_record` to see the original text.
  - DROPMALFORMED silently drops bad rows.
  - FAILFAST stops at the first bad row.
- **Spark only checks the columns a query reads**, so `count()` can miss bad rows.
- **`NOT NULL` is not enforced** when reading files. Check nulls yourself (Day 04).
- **CSV columns are matched by position.** Use `enforceSchema=False` to catch files whose columns are in the wrong order.
- **Audit columns** (`_metadata.file_path`, `current_timestamp()`) tell you where each row came from and when it was loaded.
- **Nested JSON:** a dot for a struct field (`device.os`), `["key"]` for a map, `size()` for an array.

*Skipped on purpose (they don't add a new concept): the reusable reader function, the timezone-assumption example and the automated failure check.*